# Generate a figure of log(Li/Fe) vs. Time (Gyr) using Prantzos 2012 enrichment curve

This will basically be figure 3 from the Science paper, but I'm going to use a Ca/Fe ratio to convert the log(Li/Ca) to log(Li/Fe) and plot all of the white dwarfs from the expanded sample

## 2021-10-08 The flexibility for modelers, atm_type, and overshoot have been added. Also, the decreasing phase arrow error has been corrected.

In [1]:
from __future__ import print_function

import matplotlib

matplotlib.use('pdf')
savefig=True
    
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from astropy.io import fits
from glob import glob
from astropy.time import Time
from astropy import coordinates as coords
from astropy import units as u
from astropy import constants as const
from astropy import convolution as conv
from astropy.table import Table, Column
import scipy.interpolate as scinterp
import time
import periodictable as pt

start = time.time()
print(start)
time_string=str(start).split('.')[0]

#from mendeleev import O, Ca, Li, Na, Si, Fe, Mg, He
start = time.time()

#import wdatmos
import spec_plot_tools as spt
import cal_params as cp
import plot_spec as ps
import abundance_corrections as acorr
import interp_tau as itau
import fix_strings as fs


#print(os.getcwd())

1633715518.46483
all_fwctb
(116, 4, 27)
(4, 27, 116)
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DA_diff_ov00_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DA_diff_ov10_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DB_diff_ov00_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DB_diff_ov10_diffusion_timescales.csv


In [2]:
plt.show()

In [3]:
figure_output_dir='/Users/BenKaiser/Desktop/'
#figure_output_dir='/Users/BenKaiser/Desktop/z_plots_for_Hollands/'
#figure_output_dir='/Users/BenKaiser/Desktop/GaiaJ1644m0449_paper/ApJ_reformat/figures'

In [4]:
target_dir= '/Users/BenKaiser/Desktop/radial_velocity_calculations/'
os.chdir(target_dir)

In [5]:
#wd_abund_file='all_wd_abundances.csv'
#wd_abund_file='20210131_all_wd_abundances.csv'
#wd_abund_file='20210131_all_wd_abundances_alternative.csv'
#wd_abund_file='20210607_all_wd_abundances_bedard_cooling.csv'
#wd_abund_file='20210713_all_wd_abundances_bedard_cooling_no_WDs_plot.csv'
#wd_abund_file='20210607_all_wd_abundances_bedard_cooling_test.csv'
wd_abund_file='20210818_all_wd_abundances_bedard_cooling_newer_Blouin.csv'
lodders_abund_file='Lodders2009_solarsystem_abundances.csv'
solar_system_object_file='solar_system_body_abundances.csv'
prantzos_file='Prantzos2012_fig18_approximate_datapoints.csv'

In [6]:
wd_abund_table=Table.read(wd_abund_file)
wd_abund_table=spt.clean_color_string(wd_abund_table,color_header='plot_color')
lodders_table=Table.read(lodders_abund_file)
bodies_table=Table.read(solar_system_object_file)
enrichment_table=Table.read(prantzos_file)
lodders_table.add_index('element')
wd_abund_table.add_index('name')
bodies_table.add_index('name')

limit_length=0.3 #length of limit error bars on plots
limit_indicator=99. #value above which if the absolute value of the error on a measurement is above it indicates it should be a limit



In [7]:
color_dict={
    'WDJ1644-0449':'#ff0000',
    'SDSSJ1330+6435':'#8900ff',
    'WDJ2356-209':'#00ffc5',
    'SDSSJ1636+1619':'pink',
    'WDJ2317+1830':'orange',
    'WDJ1824+1213':'g',
    'LHS2534':'b',
    'WDJ2317+1830Hollands':'orange',
    'WDJ1824+1213Hollands':'g',
    'LHS2534Hollands':'b',
    'WDJ2317+1830Blouin':'orange',
    'WDJ1824+1213Blouin':'g',
    'LHS2534Blouin':'b'
}
step_dict={
    'WDJ1644-0449':5,
    'SDSSJ1330+6435':5,
    'WDJ2356-209':5,
    'SDSSJ1636+1619':5,
    'WDJ2317+1830':5,
    'WDJ1824+1213':5,
    'LHS2534':5,
    'WDJ2317+1830Hollands':5,
    'WDJ1824+1213Hollands':5,
    'LHS2534Hollands':5,
    'WDJ2317+1830Blouin':5,
    'WDJ1824+1213Blouin':5,
    'LHS2534Blouin':5,
}

t_step=5

wd_marker='*'
#met_marker='D'
#ssp_marker='met_marker'
met_marker='s'
ssp_marker='D'
met_color='#1ca1f2'
met_size=3
ci_size=6
wd_size=10
dp_alpha=0.5
ci_leg_size=9
arr_naca=[-0.4,-0.1]
alpha_range=[0.5,0.2]
arrow_segs=100
arrow_width=0.03
#arrow_width=0.07

arrow_line=4
figure_text_size=6
default_offset=[0.05,0.00]
annot_line_weight=0.03

hide_photospheric=False
hide_decreasing=False

In [8]:
CaFe_ratio=0.2 #Ballpark value for now #This is the solar-normalized version
CaFe_ratio=CaFe_ratio+lodders_table.loc['Ca']['A_el']-lodders_table.loc['Fe']['A_el'] #Corrected to log(Ca/Fe) from [Ca/Fe] by adding the solar log(Ca/Fe)
CaFe_ratio_err=0.1 #Also ballpark estimate

Need to replace the "name" call in the plot_wd_errorbar() function to take the row directly instead of looking it up again because names can (and do) repeat now. This will most likely choke with the new defaults. It also needs to be able to correctly handle the multiple model options in the future.

In [9]:
def plot_wd_errorbar(lica,lica_err,row, selected_marker=wd_marker, markersize=wd_size, label=''):
    #if label=='':
    #    label=name
    #else:
    #    pass
    #row=wd_abund_table.loc[name]
    name=row['name']
    age_errs= [[row['age_minus']],[row['age_plus']]]
    age=row['age']
    uplims=False
    lolims=False
    xlolims=False
    xuplims=False
    print('lica_err',lica_err, type(lica_err), np.abs(lica_err))
    print('limit_indicator', limit_indicator,type(limit_indicator))
    if np.abs(lica_err)> limit_indicator:
        if lica_err > 0:
            lolims=True
        elif lica_err < 0:
            uplims=True
        else:
            print("This shouldn't print el3el2_err")
        life_err= limit_length
    else:
        #no limit indicators are present for the 2 relative abundances input
        life_err= np.sqrt(float(lica_err)**2+float(CaFe_ratio_err)**2)
    life=lica+CaFe_ratio
    
    plt.errorbar(age, life,yerr= life_err, xerr= age_errs, uplims=uplims, lolims=lolims, xuplims=xuplims, xlolims=xlolims, color=
                 row['plot_color'],marker=selected_marker,  markersize=markersize,linestyle='None')
    plt.errorbar(age,life,label=label, marker=selected_marker, markersize=markersize, color=row['plot_color'],linestyle='None')
    return

In [10]:

def plot_wd_LiFe_age(row, logg='default', teff='default', t_step=10, naca_min=-4.0, t_max=100, elements=["Li","Ca",'Na'], t_step_units='Myr',plot_type='line',SSP=True):
    """
    
    
    t_step=10, time in Myr of time-step for declining phase
    
    
    """
    name=row['name']
    print('starting plotting effort for',name)
    string1= elements[0].lower()+'/'+elements[1].lower()
    string2=elements[2].lower()+'/'+elements[1].lower()
    #times= np.arange(0, t_max+t_step, t_step)
    markersize=wd_size
    #target_row=wd_abund_table.loc[name]
    target_row=row
    target_age=target_row['age']
    target_el1el2=target_row[string1]
    target_el3el2=target_row[string2]
    el1el2_err=target_row[string1+'_err']
    el3el2_err= target_row[string2+'_err']
    label=target_row['name']
    #label=fs.fix_display_string(label)
    if logg=='default':
        logg=target_row['logg']
    else:
        pass
    if teff=='default':
        teff= target_row['teff']
    else:
        pass
    #if target_row[string1+'_err']>0.:
    #    plt.errorbar(target_row[string2],target_row[string1],xerr=target_row[string2+'_err'],yerr=target_row[string1+'_err'], marker=wd_marker, markersize=wd_size, color=color_dict[target_row['name']],linestyle='None')
    #    plt.errorbar(target_row[string2],target_row[string1], label=fs.fix_display_string(target_row['name']), marker=wd_marker, markersize=wd_size, color=color_dict[target_row['name']],linestyle='None')
    #elif target_row[string2+'_err'] < 0.001:
    #    plt.errorbar(target_row[string2],target_row[string1],xerr=0.3,yerr=0.3, uplims=True,xuplims=True, marker=wd_marker, color=color_dict[target_row['name']], markersize=wd_size,linestyle='None')
    #    plt.errorbar(target_row[string2],target_row[string1], label=fs.fix_display_string(target_row['name']), marker=wd_marker, color=color_dict[target_row['name']], markersize=wd_size,linestyle='None')
    #else:
    #    plt.errorbar(target_row[string2],target_row[string1],xerr=target_row[string2+'_err'],yerr=0.3, uplims=True, marker=wd_marker, color=color_dict[target_row['name']], markersize=wd_size,linestyle='None')
    #    plt.errorbar(target_row[string2],target_row[string1], label=fs.fix_display_string(target_row['name']), marker=wd_marker, color=color_dict[target_row['name']], markersize=wd_size,linestyle='None')
    if hide_photospheric:
        pass
    else:
        plot_wd_errorbar(target_el1el2, el1el2_err,row, selected_marker=wd_marker, markersize=wd_size, label=str(label)+ " Photospheric")
    
    
    if SSP:
        
        plot_marker=ssp_marker
        markersize=ci_size
        target_el1el2, target_el3el2, el1el2_err, el3el2_err=acorr.easy_dist_ssp(target_row,elements, plot_all=False,tau_rand=True)
        #label=label+' SSP'
        label=label+' Steady State'
    else:
        print('Not SSP!')
        plot_marker=wd_marker
        el1el2_err=target_row[string2+'_err']
        el3el2_err=target_row[string1+'_err']
        
    if t_step_units != 'Myr':
        print("t_step_units is not Myr, meaning it's some diffusion timescale multiple")
        print('so t_step= ',t_step,"* tau_",t_step_units)
        tau_time= 10.**itau.extrapolate_tau_x_logg(teff, logg, t_step_units, atm_type=target_row['diff_atm_type'], modeler=acorr.default_modeler, overshoot=acorr.default_overshoot)
        tau_time=tau_time*1e-6 #converted to Myr
        t_step=t_step*tau_time
        t_max=t_max*tau_time
        print('New t_step:',t_step, 'Myr')
        print('New t_max:', t_max, 'Myr')
    else:
        pass
    if target_row['show_dp']==1:
        if plot_type=='line':
            times= np.arange(-1*t_max-t_step, t_max+t_step, t_step)
            #dp_el1el2, dp_el3el2= acorr.el1el2_DP_el3el2_ftimes(teff, target_row['k/ca'], target_row['na/ca'],times, 'K', "Ca", "Na", logg=logg)
            dp_el1el2, dp_el3el2= acorr.el1el2_DP_el3el2_ftimes(teff, target_row[string1], target_row[string2],times, elements[0], elements[1], elements[2], logg=logg, atm_type=target_row['diff_atm_type'])
            line= plt.plot(dp_el3el2, dp_el1el2, marker='o', label="teff="+str(teff)+'K,logg='+str(logg), color=color_dict[name], alpha=dp_alpha)

            slope=(dp_el1el2-np.roll(dp_el1el2,1))/(dp_el3el2-np.roll(dp_el3el2,1))
            add_arrow(line[0],position=arr_naca[0], slope=slope[0])
            #print('Slope:', slope)
        elif (plot_type=='arrow'):
            print("successfully plot type arrow happening")
            times=t_step
            arrow_endy,arrow_endx=acorr.el1el2_DP_el3el2_ftimes(teff, target_el1el2, target_el3el2,times, elements[0], elements[1], elements[2], logg=logg, atm_type=target_row['diff_atm_type'])
            arrow_endy=arrow_endy+CaFe_ratio
            arrow_endx=target_age
            #print('arrow_endy',arrow_endy,'arrow_endx',arrow_endx)
            #plt.plot(arrow_endx,arrow_endy,marker='o')
            #ypoints=np.linspace(target_el1el2+CaFe_ratio,arrow_endy,arrow_segs)
            #xpoints=np.linspace(target_age,arrow_endx,arrow_segs)
            #dx=arrow_endx-target_el3el2
            #dy=arrow_endy-target_el1el2
            #dx=arrow_endx-xpoints[-2]
            #dy=arrow_endy-ypoints[-2]
            #def get_segs(points):
            #    return np.vstack([points,np.roll(points,1)]).T[1:]
            #x_segs=get_segs(xpoints)
            #y_segs=get_segs(ypoints)
            #print('x_segs',x_segs)
            #color_array=np.empty_like(ypoints,dtype=str)
            #color_array[:]=color_dict[name]
            #alpha_vals=np.linspace(alpha_range[0],alpha_range[1],arrow_segs)
            #print('alpha_vals',alpha_vals)
            #for x,y,alpha in zip(x_segs, y_segs,alpha_vals):
                #plt.plot(x,y,color=color_dict[name],alpha=alpha,linewidth=arrow_line)
                #plt.plot(x,y,color=color_dict[name],alpha=alpha)
            #plt.arrow(xpoints[-2],ypoints[-2],dx,dy,color=color_dict[name],width=arrow_width, alpha=alpha_range[1],length_includes_head=True )
            try:
                plt.arrow(target_age,target_el1el2+CaFe_ratio,arrow_endx-target_age,arrow_endy-(target_el1el2+CaFe_ratio),color=target_row['plot_color'],width=arrow_width, alpha=alpha_range[0],length_includes_head=True, linewidth=0 )
            except ValueError:
                print('\nPoint for',label, "can't draw arrow because no data\n")
        else:
            print('\nplot_type not recognized', plot_type,'\n')
    else:
        pass
    print('\n\n',target_row['name'])
    #plt.errorbar(target_el3el2,target_el1el2,xerr=el3el2_err,yerr=0.3, uplims=True, marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']],linestyle='None')
    #plt.errorbar(target_el3el2,target_el1el2,label=label,uplims=True, marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']],linestyle='None')
    print("about to try")
    #if target_row[elements[0].lower()+'/'+elements[1].lower()+'_err'] > 0.:
    #    plt.errorbar(target_el3el2,target_el1el2, label=fs.fix_display_string(label), marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']], linestyle='None')
    #    plt.errorbar(target_el3el2,target_el1el2,xerr=el3el2_err,yerr=el1el2_err, marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']], linestyle='None')
    #elif target_row[elements[2].lower()+'/'+elements[1].lower()+'_err'] < 0.001:
    #    plt.errorbar(target_el3el2,target_el1el2,xerr=0.3,yerr=0.3, uplims=True, xuplims=True, marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']],linestyle='None')
    #    plt.errorbar(target_el3el2,target_el1el2,label=label,uplims=True, marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']],linestyle='None')
    
    #else:
    #    plt.errorbar(target_el3el2,target_el1el2,xerr=el3el2_err,yerr=0.3, uplims=True, marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']],linestyle='None')
    #    plt.errorbar(target_el3el2,target_el1el2,label=label,uplims=True, marker=plot_marker, markersize=markersize, color=color_dict[target_row['name']],linestyle='None')
    plot_wd_errorbar(target_el1el2, el1el2_err,row, selected_marker=plot_marker, markersize=markersize, label=label)
    
    print("\n\n***\n\nPlotting concluded for", name,'\n\n***\n')
    return

In [11]:
def plot_enrichment_LiFe(age_offset=0, label='Galactic Li/Fe Enrichment Curve',color='#2B64AC',linestyle='solid',new_model_age=12):
    solar_age=4.57
    model_age=12
    ages=model_age-enrichment_table['Time(Gyr)']+age_offset
    if new_model_age != model_age:
        print("rescaling total Galactic age from", model_age, "to", new_model_age)
        change_points=np.where(ages>solar_age)
        rescale=(new_model_age-solar_age)/(model_age-solar_age)
        ages[change_points]=rescale*(ages[change_points]-solar_age)+solar_age
    else:
        pass
    #plt.plot(ages, enrichment_table['log(Li/Fe)'], label='Prantzos 2012 Enrichment curve\nfrom Fig. 18')
    plt.plot(ages, enrichment_table['log(Li/Fe)'], label=label, color=color,linestyle=linestyle)
    plt.xlim(15,0)


    plt.axvline(x=13.8, linestyle=':', color='k')
    #plt.legend(fontsize=legend_font)
    #plt.grid(True)
    plt.xlabel('Age (Gyr)')
    plt.ylabel('log(Li/Fe)')
    #plt.ylabel(r'log(Li/Fe)$_{\mathrm{Nebular}}$')
    return

In [12]:
spt.initiate_science_plot()


if hide_decreasing:
    plot_type='do not'
else:
    plot_type='arrow'
plt.figure(figsize=(7.25,7.25),constrained_layout=False)
t_max=10
for row in wd_abund_table:
    #t_step=step_dict[row['name']]
    if row['show']==0:
        pass
    elif row['show_li_evo']==0:
        pass
    else:
        #plot_wd_LiFe_age(row['name'],logg='default', t_step=t_step, naca_min=-4.0, t_max=t_max, t_step_units='Ca', plot_type=plot_type)
        plot_wd_LiFe_age(row,logg='default', t_step=t_step, naca_min=-4.0, t_max=t_max, t_step_units='Ca', plot_type=plot_type)
        #pass
    
plt.errorbar(4.57,lodders_table.loc['Li']['A_el']-lodders_table.loc['Fe']['A_el'],yerr=np.sqrt(lodders_table.loc['Li']['A_el_err']**2+lodders_table.loc['Fe']['A_el_err']**2),label="CI Chondrites",marker=met_marker, color=met_color, linestyle='None', markersize=ci_leg_size)
plt.errorbar(4.57,lodders_table.loc['Li']['A_el']-lodders_table.loc['Fe']['A_el'],yerr=np.sqrt(lodders_table.loc['Li']['A_el_err']**2+lodders_table.loc['Fe']['A_el_err']**2),marker=met_marker,markersize=ci_size, color=met_color)
plot_enrichment_LiFe()
#plot_enrichment_LiFe(age_offset=-2,label='Galactic Li/Fe Enrichment Curve (2 Gyr offset)',linestyle='--')
#plot_enrichment_LiFe(age_offset=-2,label='',linestyle='--')
#plot_enrichment_LiFe(label='',linestyle='--', new_model_age=10)
#plot_enrichment_LiFe(label='',linestyle='--', new_model_age=11.0)



plt.legend(loc='best')
plt.ylim(-4.5,0.25)

if savefig:
    print(os.getcwd())
    os.chdir(figure_output_dir)
    print(os.getcwd())
    start = time.time()
    print(start)
    time_string=str(start).split('.')[0]
    plt.savefig('LiFe_vs_age'+'_'+time_string+'.pdf')#plt.grid(True)
    print("Figure saved")
else:
    pass

plt.show()

starting plotting effort for WDJ1644-0449
lica_err 0.18 <class 'numpy.float64'> 0.18
limit_indicator 99.0 <class 'float'>
row     name     modeler cooling_model teff teff_err logg logg_err m_wd h/he h/he_err li/he li/he_err na/he na/he_err mg/he mg/he_err  k/he  k/he_err ca/he ca/he_err cr/he cr/he_err fe/he fe/he_err li/ca li/ca_err na/ca na/ca_err  k/ca k/ca_err li/na  li/na_err   k/na   k/na_err  ca/na ca/na_err  li/k   li/k_err  na/k   na/k_err  ca/fe ca/fe_err mg/fe mg/fe_err na/mg na/mg_err ca/mg ca/mg_err ca/cr ca/cr_err k/cr k/cr_err cr/fe cr/fe_err na/li  na/li_err  ca/li ca/li_err atm_type diff_atm_type log_q age age_minus age_plus   vtan_lsr  vtan_lsr_err_lo vtan_lsr_err_hi      v          uw2       v_err_lo    v_err_hi   uw2_err_lo  uw2_err_hi plot_color show show_li_evo show_geo show_dp thin_disk thick_disk halo
------------ ------- ------------- ---- -------- ---- -------- ---- ---- -------- ----- --------- ----- --------- ----- --------- ------ -------- ----- --------- -

using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
4074.3037125646074 7.979653090196411
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
4074.3037125646074 7.979653090196411
tau_Li-tau_Ca 0.5638985527812868 +/- 0.2017141889465997
tau_Na-tau_Ca 0.24044522172646388 +/- 0.1996290406567804
target_ssp Li Ca -2.8630161943101475
target_ssp Na Ca 0.8601093022108679
dist ssp Li Ca -2.8635221386417387 -2.863898552781287 0.2017141889465997
dist ssp Na Ca 0.857156897622205 0.8579576021264975 0.27991411591967075
t_step_units is not Myr, meaning it's some diffusion timescale multiple
so t_step=  5 * tau_ Ca
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
New t_step: 73.15072200118769 Myr
New t_max: 146.30144400237538 Myr
successfully plot type arrow happening
using Koester2

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:83: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order)
/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:136: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order, subok=True)


using Koester2020 models H overshoot=1.0
using Koester2020 models H overshoot=1.0
using Koester2020 models H overshoot=1.0
3488.9408799679236 7.591282396354784
using Koester2020 models H overshoot=1.0
using Koester2020 models H overshoot=1.0
using Koester2020 models H overshoot=1.0
3488.9408799679236 7.591282396354784
tau_Li-tau_Ca 0.18013853944219854 +/- 0.20277439024025487
tau_Na-tau_Ca 0.0660655356133252 +/- 0.20145406066800126
target_ssp Li Ca -2.219312255983252
target_ssp Na Ca -0.4469105776203587
dist ssp Li Ca -2.219562951149953 -2.2185868240145505 0.2998797314310685
dist ssp Na Ca -0.4475516115346958 -0.44583488746172617 0.30856227321486696
t_step_units is not Myr, meaning it's some diffusion timescale multiple
so t_step=  5 * tau_ Ca
using Koester2020 models H overshoot=1.0
using Koester2020 models H overshoot=1.0
using Koester2020 models H overshoot=1.0
New t_step: 315.5295344357394 Myr
New t_max: 631.0590688714788 Myr
successfully plot type arrow happening
using Koester2020 

using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
4994.79484160322 8.093594273104854
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
4994.79484160322 8.093594273104854
tau_Li-tau_Ca 0.5175537222838975 +/- 0.2004099947834351
tau_Na-tau_Ca 0.2192093659849438 +/- 0.20180343730151526
target_ssp Li Ca -1.9315706116121978
target_ssp Na Ca 0.4390982288533606
dist ssp Li Ca -1.923872251853802 -1.9271210836823338 0.25552022960229426
dist ssp Na Ca 0.4403994285423516 0.4389490878286254 0.23939040953226753
t_step_units is not Myr, meaning it's some diffusion timescale multiple
so t_step=  5 * tau_ Ca
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
New t_step: 25.67263591714957 Myr
New t_max: 51.34527183429914 Myr
successfully plot type arrow happening
using Koester2020

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:46: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
